# Error Analysis â€” Human Majority-Vote Reference

Systematic error analysis across all eight system configurations (decoders: zero-shot and chain-of-thought; encoders: fine-tuned) on **CSC** and **MultiPICo**. Only instances with a direct human majority-vote affordance judgment are included, and `human_majority_label` is the sole ground truth used throughout.

The analysis covers:
1. **Data loading** â€” collect only `*_human_majority.csv` predictions into a unified DataFrame
2. **All-models-wrong** â€” examples where every system fails against the human majority
3. **Input-type disagreement** â€” examples where full dialogue corrects or introduces an error
4. **Error rate by label** â€” false-positive / false-negative patterns for direct affordance judgments
5. **Model-specific error patterns** and pairwise disagreement
6. **Qualitative inspection** and paper-ready examples

In [1]:
import os
import re
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations


def find_repo_root() -> Path:
    """Find the repository root regardless of the notebook's launch directory."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "3_experiments" / "experiment1" / "output").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root from the current working directory.")


REPO_ROOT = find_repo_root()
BASE = REPO_ROOT / "3_experiments" / "experiment1" / "output"

INPUT_TYPES = ["last_u1_message", "conversation"]
COMMON_SAMPLE_INPUT_TYPES = ["context_reply", *INPUT_TYPES]
DATASETS = ["CSC", "MultiPICo"]
REFERENCE_COLUMN = "human_majority_label"

DECODERS = {
    "Gemma4-E4B": BASE / "decoders" / "google_gemma-4-E4B-it",
    "Ministral3-8B": BASE / "decoders" / "mistralai_Ministral-3-8B-Instruct-2512",
    "Qwen3.5-9B": BASE / "decoders" / "Qwen_Qwen3.5-9B",
}
DECODER_PROMPTS = ["zero_shot", "chain_of_thought"]

ENCODERS = {
    "XLM-RoBERTa-large": BASE / "encoders" / "FacebookAI_xlm-roberta-large",
    "EuroBERT-210m": BASE / "encoders" / "EuroBERT_EuroBERT-210m",
}


def read_prediction_file(path: Path) -> pd.DataFrame:
    """Read a prediction file, adding the Windows long-path prefix when needed."""
    csv_path = str(path.resolve())
    long_path_prefix = chr(92) * 2 + "?" + chr(92)
    if os.name == "nt" and not csv_path.startswith(long_path_prefix):
        csv_path = long_path_prefix + csv_path
    return pd.read_csv(
        csv_path,
        usecols=["unique_id", "model_answer", REFERENCE_COLUMN],
    )


records = []

for model_name, folder in DECODERS.items():
    for prompt in DECODER_PROMPTS:
        for dataset in DATASETS:
            for inp in COMMON_SAMPLE_INPUT_TYPES:
                candidates = list(
                    folder.glob(f"{prompt}_*_{dataset}_{inp}_human_majority.csv")
                )
                if len(candidates) != 1:
                    raise ValueError(
                        f"Expected one human-majority decoder file for "
                        f"{model_name} | {prompt} | {dataset} | {inp}; "
                        f"found {len(candidates)}"
                    )
                df = read_prediction_file(candidates[0])
                df["model"] = f"{model_name}_{prompt}"
                df["dataset"] = dataset
                df["input_type"] = inp
                records.append(df)

for model_name, enc_base in ENCODERS.items():
    for dataset in DATASETS:
        for inp in COMMON_SAMPLE_INPUT_TYPES:
            pred_dir = enc_base / dataset / "human_heldout_f1" / inp / "predictions"
            candidates = list(pred_dir.glob("*_human_majority.csv"))
            if len(candidates) != 1:
                raise ValueError(
                    f"Expected one human-majority encoder file for "
                    f"{model_name} | {dataset} | {inp}; found {len(candidates)}"
                )
            df = read_prediction_file(candidates[0])
            df["model"] = f"{model_name}_fine_tuning"
            df["dataset"] = dataset
            df["input_type"] = inp
            records.append(df)

all_preds = pd.concat(records, ignore_index=True)
all_preds["unique_id"] = all_preds["unique_id"].astype("string")
all_preds["model_answer"] = pd.to_numeric(all_preds["model_answer"], errors="coerce")
all_preds[REFERENCE_COLUMN] = pd.to_numeric(
    all_preds[REFERENCE_COLUMN], errors="coerce"
)
all_preds = all_preds.dropna(subset=["model_answer", REFERENCE_COLUMN]).copy()
all_preds["model_answer"] = all_preds["model_answer"].astype(int)
all_preds[REFERENCE_COLUMN] = all_preds[REFERENCE_COLUMN].astype(int)
all_preds["correct"] = (
    all_preds["model_answer"] == all_preds[REFERENCE_COLUMN]
).astype(int)
all_preds["error_type"] = np.select(
    [
        (all_preds["model_answer"] == 1) & (all_preds[REFERENCE_COLUMN] == 0),
        (all_preds["model_answer"] == 0) & (all_preds[REFERENCE_COLUMN] == 1),
    ],
    ["FP", "FN"],
    default="OK",
)

models = all_preds["model"].unique().tolist()
if len(models) != 8:
    raise ValueError(f"Expected 8 system configurations, found {len(models)}: {models}")

duplicate_keys = ["unique_id", "model", "dataset", "input_type"]
if all_preds.duplicated(duplicate_keys).any():
    raise ValueError("Duplicate predictions found for the same item/model/input setting.")

# Use the same per-dataset common sample as Table IV: every system must have a
# valid prediction for each reported input configuration. This also guarantees
# that all analyses compare identical human-annotated instances.
common_ids_by_dataset = {}
for dataset in DATASETS:
    valid_id_sets = []
    for model in models:
        for inp in COMMON_SAMPLE_INPUT_TYPES:
            ids = set(
                all_preds.loc[
                    (all_preds["dataset"] == dataset)
                    & (all_preds["model"] == model)
                    & (all_preds["input_type"] == inp),
                    "unique_id",
                ].tolist()
            )
            if not ids:
                raise ValueError(f"No valid predictions for {dataset} | {model} | {inp}")
            valid_id_sets.append(ids)
    common_ids_by_dataset[dataset] = set.intersection(*valid_id_sets)

    label_distribution = (
        all_preds.loc[
            (all_preds["dataset"] == dataset)
            & (all_preds["unique_id"].isin(common_ids_by_dataset[dataset])),
            ["unique_id", REFERENCE_COLUMN],
        ]
        .drop_duplicates()
        [REFERENCE_COLUMN]
        .value_counts()
        .sort_index()
        .to_dict()
    )
    print(
        f"Common human-majority sample [{dataset}]: "
        f"{len(common_ids_by_dataset[dataset])} instances | labels={label_distribution}"
    )

# Add annotator-agreement metadata for qualitative interpretation.
gold_path = REPO_ROOT / "3_experiments" / "human" / "annotations" / "gold_labels.csv"
human_gold = pd.read_csv(gold_path, dtype={"unique_id": "string"})
human_gold["gold_label"] = pd.to_numeric(human_gold["gold_label"], errors="raise").astype(int)
human_gold["unanimous"] = human_gold["unanimous"].astype(str).str.lower().eq("true")

all_preds = all_preds[all_preds["input_type"].isin(INPUT_TYPES)].copy()
all_preds = all_preds[
    all_preds.apply(
        lambda row: row["unique_id"] in common_ids_by_dataset[row["dataset"]],
        axis=1,
    )
].copy()
all_preds = all_preds.merge(
    human_gold[["unique_id", "gold_label", "unanimous"]],
    on="unique_id",
    how="left",
    validate="many_to_one",
)

if not (all_preds[REFERENCE_COLUMN] == all_preds["gold_label"]).all():
    raise ValueError("human_majority_label does not match gold_labels.csv")

print(f"Models loaded: {models}")
print(f"Total prediction rows (two context-only inputs): {len(all_preds)}")
all_preds.head(3)

Common human-majority sample [CSC]: 88 instances | labels={0: 56, 1: 32}
Common human-majority sample [MultiPICo]: 84 instances | labels={0: 61, 1: 23}
Models loaded: ['Gemma4-E4B_zero_shot', 'Gemma4-E4B_chain_of_thought', 'Ministral3-8B_zero_shot', 'Ministral3-8B_chain_of_thought', 'Qwen3.5-9B_zero_shot', 'Qwen3.5-9B_chain_of_thought', 'XLM-RoBERTa-large_fine_tuning', 'EuroBERT-210m_fine_tuning']
Total prediction rows (two context-only inputs): 2752


  unique_id  human_majority_label  ...  gold_label unanimous
0  101_1015                     1  ...           1     False
1  103_1014                     0  ...           0      True
2  103_1019                     1  ...           1      True

[3 rows x 10 columns]

## 1. Overall accuracy against the human majority by model, dataset, and input type

In [2]:
acc_table = (
    all_preds.groupby(["dataset", "input_type", "model"])["correct"]
    .mean()
    .round(3)
    .reset_index()
    .rename(columns={"correct": "accuracy"})
    .pivot_table(index=["dataset", "model"], columns="input_type", values="accuracy")
    .reset_index()
)
print(acc_table.to_string(index=False))


  dataset                          model  conversation  last_u1_message
      CSC      EuroBERT-210m_fine_tuning         0.545            0.386
      CSC    Gemma4-E4B_chain_of_thought         0.545            0.511
      CSC           Gemma4-E4B_zero_shot         0.636            0.500
      CSC Ministral3-8B_chain_of_thought         0.511            0.500
      CSC        Ministral3-8B_zero_shot         0.443            0.386
      CSC    Qwen3.5-9B_chain_of_thought         0.705            0.659
      CSC           Qwen3.5-9B_zero_shot         0.568            0.352
      CSC  XLM-RoBERTa-large_fine_tuning         0.591            0.682
MultiPICo      EuroBERT-210m_fine_tuning         0.464            0.464
MultiPICo    Gemma4-E4B_chain_of_thought         0.536            0.464
MultiPICo           Gemma4-E4B_zero_shot         0.679            0.512
MultiPICo Ministral3-8B_chain_of_thought         0.476            0.464
MultiPICo        Ministral3-8B_zero_shot         0.321          

## 2. Error-type breakdown against the human majority (FP / FN) per model Ã— dataset Ã— input type

In [3]:
error_breakdown = (
    all_preds[all_preds["error_type"] != "OK"]
    .groupby(["dataset", "input_type", "model", "error_type"])
    .size()
    .reset_index(name="count")
)
pivot_err = error_breakdown.pivot_table(
    index=["dataset", "input_type", "model"],
    columns="error_type",
    values="count",
    fill_value=0
).reset_index()
print(pivot_err.to_string(index=False))


  dataset      input_type                          model   FN   FP
      CSC    conversation      EuroBERT-210m_fine_tuning 15.0 25.0
      CSC    conversation    Gemma4-E4B_chain_of_thought  9.0 31.0
      CSC    conversation           Gemma4-E4B_zero_shot 23.0  9.0
      CSC    conversation Ministral3-8B_chain_of_thought  8.0 35.0
      CSC    conversation        Ministral3-8B_zero_shot  0.0 49.0
      CSC    conversation    Qwen3.5-9B_chain_of_thought  6.0 20.0
      CSC    conversation           Qwen3.5-9B_zero_shot 17.0 21.0
      CSC    conversation  XLM-RoBERTa-large_fine_tuning 12.0 24.0
      CSC last_u1_message      EuroBERT-210m_fine_tuning  1.0 53.0
      CSC last_u1_message    Gemma4-E4B_chain_of_thought  8.0 35.0
      CSC last_u1_message           Gemma4-E4B_zero_shot 13.0 31.0
      CSC last_u1_message Ministral3-8B_chain_of_thought  8.0 36.0
      CSC last_u1_message        Ministral3-8B_zero_shot  2.0 52.0
      CSC last_u1_message    Qwen3.5-9B_chain_of_thought 11.0 

## 3. Cases where all systems are wrong against the human majority

For each dataset and input type, identify examples where every system disagrees with the direct majority-vote affordance judgment. The `unanimous` field indicates whether all three human annotators supplied the same judgment.

In [4]:
all_wrong_by_setting = {}

for dataset in DATASETS:
    for inp in INPUT_TYPES:
        subset = all_preds[
            (all_preds["dataset"] == dataset)
            & (all_preds["input_type"] == inp)
        ]
        n_models = subset["model"].nunique()
        wrong_counts = (
            subset[subset["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_models_wrong")
        )
        reference_map = subset[
            ["unique_id", REFERENCE_COLUMN, "unanimous"]
        ].drop_duplicates()
        wrong_counts = wrong_counts.merge(reference_map, on="unique_id")
        all_wrong = wrong_counts[
            wrong_counts["n_models_wrong"] == n_models
        ].copy()
        all_wrong["dataset"] = dataset
        all_wrong["input_type"] = inp
        all_wrong["n_total_models"] = n_models
        key = f"{dataset}_{inp}"
        all_wrong_by_setting[key] = all_wrong

        n_negative = (all_wrong[REFERENCE_COLUMN] == 0).sum()
        n_positive = (all_wrong[REFERENCE_COLUMN] == 1).sum()
        n_unanimous = all_wrong["unanimous"].sum()
        print(
            f"[{key}] total models={n_models} | all wrong={len(all_wrong)} | "
            f"human label 0={n_negative}, label 1={n_positive} | "
            f"unanimous={n_unanimous}/{len(all_wrong)}"
        )

[CSC_last_u1_message] total models=8 | all wrong=2 | human label 0=2, label 1=0 | unanimous=2/2
[CSC_conversation] total models=8 | all wrong=0 | human label 0=0, label 1=0 | unanimous=0/0
[MultiPICo_last_u1_message] total models=8 | all wrong=4 | human label 0=4, label 1=0 | unanimous=4/4
[MultiPICo_conversation] total models=8 | all wrong=1 | human label 0=1, label 1=0 | unanimous=1/1


## 4. Input-type disagreement against the human majority

For each system, find examples where it is wrong using only the last message but correct using the full conversation, and vice versa.

In [5]:
# Each row represents one model, dataset, and item, with correctness under both inputs.
pivot_inp = all_preds.pivot_table(
    index=["model", "dataset", "unique_id", REFERENCE_COLUMN, "unanimous"],
    columns="input_type",
    values="correct",
).reset_index()

pivot_inp = pivot_inp.dropna(subset=INPUT_TYPES)
pivot_inp[INPUT_TYPES] = pivot_inp[INPUT_TYPES].astype(int)

pivot_inp["lu1_wrong_conv_ok"] = (
    (pivot_inp["last_u1_message"] == 0)
    & (pivot_inp["conversation"] == 1)
).astype(int)
pivot_inp["lu1_ok_conv_wrong"] = (
    (pivot_inp["last_u1_message"] == 1)
    & (pivot_inp["conversation"] == 0)
).astype(int)
pivot_inp["both_wrong"] = (
    (pivot_inp["last_u1_message"] == 0)
    & (pivot_inp["conversation"] == 0)
).astype(int)

for dataset in DATASETS:
    sub = pivot_inp[pivot_inp["dataset"] == dataset]
    print(f"\n=== {dataset} ===")
    summary = sub.groupby("model")[[
        "lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"
    ]].sum()
    print(summary.to_string())


=== CSC ===
input_type                      lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                           
EuroBERT-210m_fine_tuning                      31                 17          23
Gemma4-E4B_chain_of_thought                    17                 14          26
Gemma4-E4B_zero_shot                           31                 19          13
Ministral3-8B_chain_of_thought                 17                 16          27
Ministral3-8B_zero_shot                         7                  2          47
Qwen3.5-9B_chain_of_thought                    18                 14          12
Qwen3.5-9B_zero_shot                           35                 16          22
XLM-RoBERTa-large_fine_tuning                   9                 17          19

=== MultiPICo ===
input_type                      lu1_wrong_conv_ok  lu1_ok_conv_wrong  both_wrong
model                                                                        

### 4b. Aggregate across systems: beneficial, harmful, and net context effects

For each item, count how many systems are corrected by the full dialogue (`last-message wrong ? conversation correct`) and how many exhibit the inverse pattern (`last-message correct ? conversation wrong`). We report effects involving at least one system, the net balance across all eight systems, and strict-majority effects separately. This distinction prevents the absence of majority-level harm from being interpreted as the absence of harmful context effects.

In [6]:
agg_disagree = (
    pivot_inp.groupby(
        ["dataset", "unique_id", REFERENCE_COLUMN, "unanimous"]
    )[["lu1_wrong_conv_ok", "lu1_ok_conv_wrong", "both_wrong"]]
    .sum()
    .reset_index()
)

n_total_models = pivot_inp["model"].nunique()
agg_disagree["net_context_effect"] = (
    agg_disagree["lu1_wrong_conv_ok"]
    - agg_disagree["lu1_ok_conv_wrong"]
)

# Strict-majority effects: at least five of the eight systems move in the same direction.
context_helps = agg_disagree[
    agg_disagree["lu1_wrong_conv_ok"] > n_total_models / 2
].copy()
context_hurts = agg_disagree[
    agg_disagree["lu1_ok_conv_wrong"] > n_total_models / 2
].copy()

# Broader item-level effects.
context_helps_any = agg_disagree[
    agg_disagree["lu1_wrong_conv_ok"] > 0
].copy()
context_hurts_any = agg_disagree[
    agg_disagree["lu1_ok_conv_wrong"] > 0
].copy()
context_net_helps = agg_disagree[
    agg_disagree["net_context_effect"] > 0
].copy()
context_net_hurts = agg_disagree[
    agg_disagree["net_context_effect"] < 0
].copy()
context_net_ties = agg_disagree[
    agg_disagree["net_context_effect"] == 0
].copy()

for dataset in DATASETS:
    ds_all = agg_disagree[agg_disagree["dataset"] == dataset]
    ch = context_helps[context_helps["dataset"] == dataset]
    cu = context_hurts[context_hurts["dataset"] == dataset]
    ch_any = context_helps_any[context_helps_any["dataset"] == dataset]
    cu_any = context_hurts_any[context_hurts_any["dataset"] == dataset]
    net_help = context_net_helps[context_net_helps["dataset"] == dataset]
    net_hurt = context_net_hurts[context_net_hurts["dataset"] == dataset]
    net_tie = context_net_ties[context_net_ties["dataset"] == dataset]

    print(f"\n=== {dataset} (total systems={n_total_models}, items={len(ds_all)}) ===")
    print(
        f"  At least one system: context helps={len(ch_any)}, "
        f"context hurts={len(cu_any)}"
    )
    print(
        f"  Net item-level effect: helps={len(net_help)}, "
        f"hurts={len(net_hurt)}, tie={len(net_tie)}"
    )
    print(
        f"  Strict-majority effect: helps={len(ch)}, hurts={len(cu)}"
    )
    print(
        f"  Maximum systems hurt on one item: "
        f"{int(ds_all['lu1_ok_conv_wrong'].max())}/{n_total_models}"
    )
    print(
        f"  Majority-help labels: label 0={(ch[REFERENCE_COLUMN] == 0).sum()}, "
        f"label 1={(ch[REFERENCE_COLUMN] == 1).sum()}"
    )


=== CSC (total systems=8, items=88) ===
  At least one system: context helps=73, context hurts=67
  Net item-level effect: helps=49, hurts=33, tie=6
  Strict-majority effect: helps=5, hurts=0
  Maximum systems hurt on one item: 4/8
  Majority-help labels: label 0=4, label 1=1

=== MultiPICo (total systems=8, items=84) ===
  At least one system: context helps=65, context hurts=57
  Net item-level effect: helps=37, hurts=28, tie=19
  Strict-majority effect: helps=0, hurts=0
  Maximum systems hurt on one item: 4/8
  Majority-help labels: label 0=0, label 1=0


## 5. System-unique errors against the human majority

In [7]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        # per example: count how many models got it wrong
        wrong_counts = (
            sub[sub["correct"] == 0]
            .groupby("unique_id")["model"]
            .nunique()
            .reset_index(name="n_wrong_models")
        )
        # unique errors = only 1 model wrong
        unique_errors = wrong_counts[wrong_counts["n_wrong_models"] == 1]["unique_id"].tolist()
        # which model is responsible?
        resp = (
            sub[(sub["correct"] == 0) & (sub["unique_id"].isin(unique_errors))]
            .groupby("model")["unique_id"]
            .count()
            .sort_values(ascending=False)
        )
        print(f"\n[{dataset} | {inp}]  unique errors per model:")
        print(resp.to_string())



[CSC | last_u1_message]  unique errors per model:
model
Gemma4-E4B_chain_of_thought      1
Gemma4-E4B_zero_shot             1
Qwen3.5-9B_chain_of_thought      1
Qwen3.5-9B_zero_shot             1
XLM-RoBERTa-large_fine_tuning    1

[CSC | conversation]  unique errors per model:
model
Ministral3-8B_zero_shot           2
Gemma4-E4B_chain_of_thought       1
Gemma4-E4B_zero_shot              1
Ministral3-8B_chain_of_thought    1
Qwen3.5-9B_chain_of_thought       1
Qwen3.5-9B_zero_shot              1

[MultiPICo | last_u1_message]  unique errors per model:
model
Gemma4-E4B_zero_shot              3
XLM-RoBERTa-large_fine_tuning     2
EuroBERT-210m_fine_tuning         1
Ministral3-8B_chain_of_thought    1
Ministral3-8B_zero_shot           1
Qwen3.5-9B_zero_shot              1

[MultiPICo | conversation]  unique errors per model:
model
Gemma4-E4B_zero_shot              4
Ministral3-8B_zero_shot           2
EuroBERT-210m_fine_tuning         1
Ministral3-8B_chain_of_thought    1
Qwen3.5-9B_zero

## 6. Pairwise model disagreement matrix

For each pair of models (same dataset + input_type), what fraction of examples do they predict differently?

In [8]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        sub = all_preds[(all_preds["dataset"] == dataset) & (all_preds["input_type"] == inp)]
        wide = sub.pivot_table(index="unique_id", columns="model", values="model_answer")
        wide = wide.dropna()
        model_cols = wide.columns.tolist()
        rows = []
        for m1, m2 in combinations(model_cols, 2):
            disagree_rate = (wide[m1] != wide[m2]).mean()
            rows.append({"model_A": m1, "model_B": m2, "disagree_rate": round(disagree_rate, 3)})
        df_dis = pd.DataFrame(rows).sort_values("disagree_rate", ascending=False)
        print(f"\n=== {dataset} | {inp} ===")
        print(df_dis.to_string(index=False))



=== CSC | last_u1_message ===
                       model_A                        model_B  disagree_rate
     EuroBERT-210m_fine_tuning    Qwen3.5-9B_chain_of_thought          0.545
   Qwen3.5-9B_chain_of_thought           Qwen3.5-9B_zero_shot          0.534
   Gemma4-E4B_chain_of_thought  XLM-RoBERTa-large_fine_tuning          0.511
   Gemma4-E4B_chain_of_thought    Qwen3.5-9B_chain_of_thought          0.511
          Qwen3.5-9B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.511
          Gemma4-E4B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.500
     EuroBERT-210m_fine_tuning  XLM-RoBERTa-large_fine_tuning          0.500
       Ministral3-8B_zero_shot  XLM-RoBERTa-large_fine_tuning          0.500
       Ministral3-8B_zero_shot    Qwen3.5-9B_chain_of_thought          0.477
          Gemma4-E4B_zero_shot    Qwen3.5-9B_chain_of_thought          0.455
          Gemma4-E4B_zero_shot           Qwen3.5-9B_zero_shot          0.443
     EuroBERT-210m_fine_tuning           Gemm

## 7. Qualitative inspection â€” load original dialogues and display hard cases

In [9]:
import json

CORPUS_ROOT = REPO_ROOT / "3_experiments" / "datasets"


def load_json_dataset(path):
    with open(path, encoding="utf-8") as stream:
        data = json.load(stream)
    entries = data["entries"] if "entries" in data else data
    rows = []
    for entry in entries:
        rows.append(
            {
                "unique_id": str(entry.get("unique_id")),
                "post": entry.get("post"),
                "reply": entry.get("reply"),
                "context": entry.get("context"),
                "dialogue": str(entry.get("dialogue", "")),
                "source": entry.get("source"),
            }
        )
    frame = pd.DataFrame(rows)
    return frame.merge(
        human_gold[["unique_id", "gold_label", "unanimous"]],
        on="unique_id",
        how="left",
        validate="one_to_one",
    )


csc_df = load_json_dataset(
    CORPUS_ROOT / "CSC_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json"
)
multipico_df = load_json_dataset(
    CORPUS_ROOT / "MultiPICo_meta-llama_Llama-3.1-8B-Instruct_n_turn_2_final.json"
)

corpus = {"CSC": csc_df, "MultiPICo": multipico_df}
print("CSC rows:", len(csc_df), "| human-labeled:", csc_df["gold_label"].notna().sum())
print(
    "MultiPICo rows:", len(multipico_df),
    "| human-labeled:", multipico_df["gold_label"].notna().sum(),
)

CSC rows: 2905 | human-labeled: 92
MultiPICo rows: 1377 | human-labeled: 85


In [10]:
def show_examples(uid_list, dataset_name, n=5, title=""):
    """Print dialogue text and direct human ground truth for selected IDs."""
    df_src = corpus[dataset_name]
    text_cols = ["post", "reply", "context", "dialogue"]
    sample = list(uid_list)[:n]
    print(f"\n{'=' * 78}")
    print(f"  {title} [{dataset_name}] showing {len(sample)}/{len(uid_list)}")
    print(f"{'=' * 78}")
    for uid in sample:
        row = df_src[df_src["unique_id"] == str(uid)]
        if row.empty:
            print(f"\n  [{uid}] (not found in corpus)")
            continue
        row = row.iloc[0]
        print(
            f"\n  ID: {uid} | human_majority_label={row['gold_label']} "
            f"| unanimous={row['unanimous']}"
        )
        for column in text_cols:
            value = row.get(column, "")
            if pd.notna(value) and str(value).strip():
                print(f"  {column}: {str(value)[:700]}")
        print()


def show_prediction_transitions(unique_id, dataset_name):
    """Display each system's prediction under the two context-only inputs."""
    rows = all_preds[
        (all_preds["dataset"] == dataset_name)
        & (all_preds["unique_id"] == str(unique_id))
    ]
    wide = rows.pivot(
        index="model", columns="input_type", values="model_answer"
    )[INPUT_TYPES]
    wide["human_majority_label"] = rows[REFERENCE_COLUMN].iloc[0]
    print(wide.to_string())

### 7a. All-system errors under both context-only input settings

In [11]:
for dataset in DATASETS:
    for inp in INPUT_TYPES:
        key = f"{dataset}_{inp}"
        uids = all_wrong_by_setting[key]["unique_id"].tolist()
        show_examples(uids, dataset, n=10, title=f"ALL SYSTEMS WRONG | {inp}")


  ALL SYSTEMS WRONG | last_u1_message [CSC] showing 2/2

  ID: 209_2066 | human_majority_label=0.0 | unanimous=True
  reply: What do you see as the main problem with him?
  context: Steve has a brother Bill. Bill often feels annoyed by his friend. The reasons that Bill finds his friend annoying are the same as the reasons why you find Steve annoying (for example, both Steve and Bill's friend always ask for money and never pay it back).  Steve says, "why is my brother even friends with that guy? I don't get it."
  dialogue: {'turns': [{'u1': "I'm telling you, it's like he's draining our whole crew of all our money. Does he ever give anyone back?", 'u2': "That is part of it, but it's not just about money. He can be really needy and whiny, too."}, {'u1': "You know, I'm starting to think that he's just a huge time waster. We're always trying to hang out, but he's always coming up with some lame excuse.", 'u2': "Yeah, that's definitely a problem. And it's not just the time wasting – it's t

### 7b. Cases where full dialogue helps a majority of systems

In [12]:
for dataset in DATASETS:
    uids = context_helps[context_helps["dataset"] == dataset]["unique_id"].tolist()
    show_examples(uids, dataset, n=5, title="CONTEXT HELPS (lu1 wrong → conv right)")



  CONTEXT HELPS (lu1 wrong → conv right) [CSC] showing 5/5

  ID: 109_1012 | human_majority_label=0.0 | unanimous=True
  reply: Nobody's going to be 100 percent compatible with you, there's no 1, just a .67 or a .74 or if you're really lucky, a .87 that you round up to 1, nobody's perfect.
  context: Steve's girlfriend is totally amazing. She is sweet, beautiful, funny, and cares a lot about Steve.  And yet, Steve is always making silly complaints about her.  Steve says, "I mean, she is great in many other ways, but her music taste is just horrible."
  dialogue: {'turns': [{'u1': "I mean, I know she's great in a lot of other ways, but this music taste thing is really getting to me.", 'u2': 'You’re being a bit too hard on her, aren’t you? She might have some questionable tastes, but she’s clearly trying to connect with you.'}, {'u1': "I guess I'm just expecting more from her, you know, since she's so great in other ways.", 'u2': "Expecting perfection is a recipe for disappointment, Ste

### 7c. Paper-ready examples of beneficial and harmful context effects

The examples below have unanimous human judgments. The suitcase and self-isolation cases illustrate context correcting predictions in opposite label directions; the social-media-help case illustrates the inverse pattern, where four systems lose a previously correct prediction after reading the full dialogue.

In [13]:
paper_examples = {
    "CSC": ["131_1020", "103_1031"],
    "MultiPICo": ["12750553915393859827_625067732195999980"],
}

for dataset, uids in paper_examples.items():
    show_examples(uids, dataset, n=len(uids), title="PAPER-READY CONTEXT EFFECT")
    for uid in uids:
        print(f"\nPrediction transitions for {uid}:")
        show_prediction_transitions(uid, dataset)


  PAPER-READY CONTEXT EFFECT [CSC] showing 2/2

  ID: 131_1020 | human_majority_label=1.0 | unanimous=True
  reply: When I get my charger and toaster back, yes
  context: Steve borrowed your spare phone charger two months ago. Then he took your toaster a month ago. He did not return any of them.  And now, Steve says, "can I borrow your suitcase? I need one for my trip next week."
  dialogue: {'turns': [{'u1': "Hey man, I've been meaning to return some stuff to you, but I've been really busy. I think it's been a couple of months now, right? Anyway, can I get some space to return them this weekend?", 'u2': "Steve, we talked about this last week. I need those items back, especially the charger. It's not a matter of giving you space, it's about getting your own stuff back. When are you bringing them over?"}, {'u1': "Look, I know I haven't returned your toaster yet, but I'll try to bring it over this weekend. Does that sound good to you?", 'u2': "Don't make promises you don't intend to kee

## 8. Export human-majority hard cases for manual inspection

Exports are written to a dedicated subdirectory so that they cannot be confused with proxy-reference analyses.

In [14]:
OUT_DIR = (
    REPO_ROOT
    / "3_experiments"
    / "experiment1"
    / "error_analysis_output"
    / "human_majority"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)


def enrich_and_save(uid_df, dataset_name, filename):
    """Merge aggregate results with text and direct human-label metadata."""
    df_src = corpus[dataset_name]
    text_cols = [
        "unique_id", "gold_label", "unanimous", "post", "reply", "context", "dialogue"
    ]
    base = uid_df.drop(columns=["unanimous"], errors="ignore")
    merged = base.merge(
        df_src[text_cols].drop_duplicates("unique_id"),
        on="unique_id",
        how="left",
        validate="one_to_one",
    )
    path = OUT_DIR / filename
    merged.to_csv(path, index=False)
    print(f"Saved {len(merged)} rows -> {path}")
    return merged


def remove_stale_export(filename):
    """Remove an export from an earlier run when its category is now empty."""
    path = OUT_DIR / filename
    if path.exists():
        path.unlink()
        print(f"Removed stale export: {path}")


for dataset in DATASETS:
    for inp in INPUT_TYPES:
        key = f"{dataset}_{inp}"
        data = all_wrong_by_setting[key]
        filename = f"all_wrong_{dataset}_{inp}_human_majority.csv"
        if not data.empty:
            enrich_and_save(
                data[["unique_id", REFERENCE_COLUMN, "n_models_wrong"]],
                dataset,
                filename,
            )
        else:
            remove_stale_export(filename)

for dataset in DATASETS:
    helps = context_helps[context_helps["dataset"] == dataset]
    helps_filename = f"context_helps_{dataset}_human_majority.csv"
    if not helps.empty:
        enrich_and_save(
            helps[["unique_id", REFERENCE_COLUMN, "lu1_wrong_conv_ok"]],
            dataset,
            helps_filename,
        )
    else:
        remove_stale_export(helps_filename)

    hurts = context_hurts[context_hurts["dataset"] == dataset]
    hurts_filename = f"context_hurts_{dataset}_human_majority.csv"
    if not hurts.empty:
        enrich_and_save(
            hurts[["unique_id", REFERENCE_COLUMN, "lu1_ok_conv_wrong"]],
            dataset,
            hurts_filename,
        )
    else:
        remove_stale_export(hurts_filename)

    net_hurts = context_net_hurts[context_net_hurts["dataset"] == dataset]
    net_hurts_filename = f"context_net_hurts_{dataset}_human_majority.csv"
    if not net_hurts.empty:
        enrich_and_save(
            net_hurts[[
                "unique_id", REFERENCE_COLUMN, "lu1_wrong_conv_ok",
                "lu1_ok_conv_wrong", "net_context_effect",
            ]],
            dataset,
            net_hurts_filename,
        )
    else:
        remove_stale_export(net_hurts_filename)

print("\nAll human-majority exports done.")

Saved 2 rows -> C:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\human_majority\all_wrong_CSC_last_u1_message_human_majority.csv
Saved 4 rows -> C:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\human_majority\all_wrong_MultiPICo_last_u1_message_human_majority.csv
Saved 1 rows -> C:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\human_majority\all_wrong_MultiPICo_conversation_human_majority.csv
Saved 5 rows -> C:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\human_majority\context_helps_CSC_human_majority.csv
Saved 33 rows -> C:\Users\micha\OneDrive\Desktop\Università\Progetti\appropriateness-of-irony\3_experiments\experiment1\error_analysis_output\human_majority\context_net_hurts_CSC_huma